In [2]:
!pip install transformers datasets accelerate bitsandbytes sentencepiece -q
print("✅ All packages installed!")

✅ All packages installed!


In [ ]:
import os
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("✅ Logged in to HuggingFace!")

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n📦 Loading TinyLlama model...")
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ TinyLlama loaded!")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

print("\n📦 Loading FULL dataset...")
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = dataset["train"].to_pandas()
print(f"✅ Dataset loaded: {len(df)} records")
print(f"   Categories: {df['category'].nunique()}")
print(f"   Intents: {df['intent'].nunique()}")

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4090
GPU Memory: 25.3 GB

📦 Loading TinyLlama model...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

✅ TinyLlama loaded!
   Parameters: 1,100,048,384

📦 Loading FULL dataset...


README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

✅ Dataset loaded: 26872 records
   Categories: 11
   Intents: 27


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import numpy as np

class ChatDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.data = []
        print("⚙️ Tokenizing full dataset...")
        for i, (_, row) in enumerate(df.iterrows()):
            text = f"""<|system|>
You are a helpful customer support assistant for an e-commerce platform.
</s>
<|user|>
{row['instruction']}
</s>
<|assistant|>
{row['response']}
</s>"""
            enc = tokenizer(text, truncation=True, max_length=max_len,
                          padding="max_length", return_tensors="pt")
            self.data.append({
                "input_ids"     : enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels"        : enc["input_ids"].squeeze()
            })
            if (i+1) % 5000 == 0:
                print(f"  Tokenized {i+1}/{len(df)} records...")
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

print("📦 Preparing full dataset...")
train_size    = int(len(df) * 0.9)
train_df      = df[:train_size]
val_df        = df[train_size:]
print(f"  Train: {len(train_df)} | Val: {len(val_df)}")

train_dataset = ChatDataset(train_df, tokenizer)
val_dataset   = ChatDataset(val_df,   tokenizer)

train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=8, shuffle=False)
print(f"✅ Dataset ready!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches  : {len(val_loader)}")

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

# Training
print("\n🚀 Starting fine-tuning on FULL dataset...")
print("="*60)

history = []
for epoch in range(3):
    total_loss    = 0
    valid_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=input_ids,
                           attention_mask=attention_mask,
                           labels=labels)
            loss = outputs.loss

        if torch.isnan(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss    += loss.item()
        valid_batches += 1

        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} "
                  f"| Loss: {loss.item():.4f}")

        if batch_idx % 200 == 0:
            torch.cuda.empty_cache()

    avg_loss = total_loss / max(valid_batches, 1)
    history.append(avg_loss)
    print(f"\n{'='*60}")
    print(f"  ✅ Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")
    print(f"{'='*60}\n")

print("✅ Fine-tuning Complete!")
print(f"   Loss history: {[round(l,4) for l in history]}")

# Save model
model.save_pretrained("/workspace/tinyllama-finetuned")
tokenizer.save_pretrained("/workspace/tinyllama-finetuned")
print("✅ Model saved to /workspace/tinyllama-finetuned!")

📦 Preparing full dataset...
  Train: 24184 | Val: 2688
⚙️ Tokenizing full dataset...
  Tokenized 5000/24184 records...
  Tokenized 10000/24184 records...
  Tokenized 15000/24184 records...
  Tokenized 20000/24184 records...
⚙️ Tokenizing full dataset...
✅ Dataset ready!
   Train batches: 3023
   Val batches  : 336

🚀 Starting fine-tuning on FULL dataset...

  ✅ Epoch 1 Complete | Avg Loss: 7.8205


  ✅ Epoch 2 Complete | Avg Loss: 0.0000


  ✅ Epoch 3 Complete | Avg Loss: 0.0000

✅ Fine-tuning Complete!
   Loss history: [7.8205, 0.0, 0.0]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /workspace/tinyllama-finetuned!


In [6]:
# Clean dataset to fix NaN issue
df_clean = df[
    (df['response'].str.len() < 300) &
    (df['instruction'].str.len() < 150) &
    (df['response'].notna()) &
    (df['instruction'].notna())
].reset_index(drop=True)

print(f"Original : {len(df)} records")
print(f"Cleaned  : {len(df_clean)} records")
print(f"Removed  : {len(df) - len(df_clean)} records")
print(f"\nCategory distribution:")
print(df_clean['category'].value_counts())

Original : 26872 records
Cleaned  : 1756 records
Removed  : 25116 records

Category distribution:
category
CANCEL          629
SHIPPING        566
INVOICE         220
CONTACT         114
ORDER           103
REFUND           43
ACCOUNT          35
FEEDBACK         16
DELIVERY         14
PAYMENT           9
SUBSCRIPTION      7
Name: count, dtype: int64


In [7]:
# Relaxed cleaning
df_clean = df[
    (df['response'].str.len() < 800) &
    (df['instruction'].str.len() < 300) &
    (df['response'].notna()) &
    (df['instruction'].notna())
].reset_index(drop=True)

print(f"Original : {len(df)} records")
print(f"Cleaned  : {len(df_clean)} records")
print(f"Removed  : {len(df) - len(df_clean)} records")
print(f"\nCategory distribution:")
print(df_clean['category'].value_counts())

Original : 26872 records
Cleaned  : 20890 records
Removed  : 5982 records

Category distribution:
category
ACCOUNT         3964
ORDER           2932
INVOICE         1951
CONTACT         1906
REFUND          1904
FEEDBACK        1884
SHIPPING        1485
PAYMENT         1479
DELIVERY        1459
SUBSCRIPTION     976
CANCEL           950
Name: count, dtype: int64


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

class ChatDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.data = []
        print(f"⚙️ Tokenizing {len(df)} records...")
        for i, (_, row) in enumerate(df.iterrows()):
            text = f"""<|system|>
You are a helpful e-commerce customer support assistant.
</s>
<|user|>
{row['instruction']}
</s>
<|assistant|>
{row['response']}
</s>"""
            enc = tokenizer(text, truncation=True, max_length=max_len,
                          padding="max_length", return_tensors="pt")
            self.data.append({
                "input_ids"     : enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels"        : enc["input_ids"].squeeze()
            })
            if (i+1) % 5000 == 0:
                print(f"  Tokenized {i+1}/{len(df)} records...")
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# Split data
train_size    = int(len(df_clean) * 0.9)
train_df      = df_clean[:train_size]
val_df        = df_clean[train_size:]
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# Create datasets
train_dataset = ChatDataset(train_df, tokenizer)
val_dataset   = ChatDataset(val_df,   tokenizer)

train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=8, shuffle=False)
print(f"✅ Dataset ready!")
print(f"   Train batches: {len(train_loader)}")

# Reload model fresh
print("\n🔄 Reloading fresh model...")
from transformers import AutoModelForCausalLM
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)

device    = torch.device("cuda")
optimizer = AdamW(model.parameters(), lr=2e-5)

# Scheduler to reduce LR automatically
total_steps = len(train_loader) * 3
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps  = total_steps // 10,
    num_training_steps= total_steps
)

model.train()
print("\n🚀 Starting fine-tuning on clean dataset...")
print("="*60)

history = []
for epoch in range(3):
    total_loss    = 0
    valid_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=input_ids,
                           attention_mask=attention_mask,
                           labels=labels)
            loss = outputs.loss

        # Skip NaN or exploding losses
        if torch.isnan(loss) or loss.item() > 10:
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()  # ← reduces LR automatically

        total_loss    += loss.item()
        valid_batches += 1

        if (batch_idx + 1) % 100 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} "
                  f"| Loss: {loss.item():.4f} | LR: {current_lr:.2e}")

        if batch_idx % 200 == 0:
            torch.cuda.empty_cache()

    avg_loss = total_loss / max(valid_batches, 1)
    history.append(avg_loss)
    print(f"\n{'='*60}")
    print(f"  ✅ Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")
    print(f"{'='*60}\n")

print("✅ Fine-tuning Complete!")
print(f"   Loss history: {[round(l,4) for l in history]}")

model.save_pretrained("/workspace/tinyllama-finetuned-clean")
tokenizer.save_pretrained("/workspace/tinyllama-finetuned-clean")
print("✅ Model saved to /workspace/tinyllama-finetuned-clean!")

Train: 18801 | Val: 2089
⚙️ Tokenizing 18801 records...
  Tokenized 5000/18801 records...
  Tokenized 10000/18801 records...
  Tokenized 15000/18801 records...
⚙️ Tokenizing 2089 records...
✅ Dataset ready!
   Train batches: 2351

🔄 Reloading fresh model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🚀 Starting fine-tuning on clean dataset...

  ✅ Epoch 1 Complete | Avg Loss: 3.1034


  ✅ Epoch 2 Complete | Avg Loss: 0.0000


  ✅ Epoch 3 Complete | Avg Loss: 0.0000

✅ Fine-tuning Complete!
   Loss history: [3.1034, 0.0, 0.0]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /workspace/tinyllama-finetuned-clean!


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import get_linear_schedule_with_warmup

torch.cuda.empty_cache()

# Load GPT-2 instead
print("📦 Loading GPT-2 model...")
model_name = "gpt2-medium"
tokenizer2 = GPT2Tokenizer.from_pretrained(model_name)
tokenizer2.pad_token = tokenizer2.eos_token

model2 = GPT2LMHeadModel.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # float32 = stable!
)
device = torch.device("cuda")
model2 = model2.to(device)
print(f"✅ GPT-2 loaded!")
print(f"   Parameters: {sum(p.numel() for p in model2.parameters()):,}")

class ChatDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.data = []
        print(f"⚙️ Tokenizing {len(df)} records...")
        for i, (_, row) in enumerate(df.iterrows()):
            text = f"Customer: {row['instruction']} Support: {row['response']}{tokenizer.eos_token}"
            enc  = tokenizer(text, truncation=True, max_length=max_len,
                            padding="max_length", return_tensors="pt")
            self.data.append({
                "input_ids"     : enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels"        : enc["input_ids"].squeeze()
            })
            if (i+1) % 5000 == 0:
                print(f"  Tokenized {i+1}/{len(df)}...")
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# Split
train_size    = int(len(df_clean) * 0.9)
train_df      = df_clean[:train_size]
val_df        = df_clean[train_size:]

train_dataset = ChatDataset(train_df, tokenizer2)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True)
print(f"✅ Dataset ready! Train batches: {len(train_loader)}")

# Optimizer & scheduler
optimizer   = AdamW(model2.parameters(), lr=3e-5)
total_steps = len(train_loader) * 3
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps  = 100,
    num_training_steps= total_steps
)

model2.train()
print("\n🚀 Starting GPT-2 fine-tuning...")
print("="*60)

history = []
for epoch in range(3):
    total_loss    = 0
    valid_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model2(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss

        if torch.isnan(loss) or loss.item() > 15:
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss    += loss.item()
        valid_batches += 1

        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} "
                  f"| Loss: {loss.item():.4f}")

        if batch_idx % 200 == 0:
            torch.cuda.empty_cache()

    avg_loss = total_loss / max(valid_batches, 1)
    history.append(avg_loss)
    print(f"\n{'='*60}")
    print(f"  ✅ Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")
    print(f"{'='*60}\n")

print("✅ Fine-tuning Complete!")
print(f"   Loss history: {[round(l,4) for l in history]}")

model2.save_pretrained("/workspace/gpt2-finetuned")
tokenizer2.save_pretrained("/workspace/gpt2-finetuned")
print("✅ GPT-2 model saved!")

📦 Loading GPT-2 model...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

✅ GPT-2 loaded!
   Parameters: 354,823,168
⚙️ Tokenizing 18801 records...
  Tokenized 5000/18801...
  Tokenized 10000/18801...
  Tokenized 15000/18801...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


✅ Dataset ready! Train batches: 2351

🚀 Starting GPT-2 fine-tuning...
  Epoch 1 | Batch 100/2351 | Loss: 1.0852
  Epoch 1 | Batch 200/2351 | Loss: 1.1708
  Epoch 1 | Batch 300/2351 | Loss: 0.7523
  Epoch 1 | Batch 400/2351 | Loss: 1.0752
  Epoch 1 | Batch 500/2351 | Loss: 0.8646
  Epoch 1 | Batch 600/2351 | Loss: 0.7585
  Epoch 1 | Batch 700/2351 | Loss: 0.8514
  Epoch 1 | Batch 800/2351 | Loss: 1.0055
  Epoch 1 | Batch 900/2351 | Loss: 0.8206
  Epoch 1 | Batch 1000/2351 | Loss: 0.8126
  Epoch 1 | Batch 1100/2351 | Loss: 0.8289
  Epoch 1 | Batch 1200/2351 | Loss: 0.6713
  Epoch 1 | Batch 1300/2351 | Loss: 0.8328
  Epoch 1 | Batch 1400/2351 | Loss: 0.6749
  Epoch 1 | Batch 1500/2351 | Loss: 0.6387
  Epoch 1 | Batch 1600/2351 | Loss: 0.7351
  Epoch 1 | Batch 1700/2351 | Loss: 0.7345
  Epoch 1 | Batch 1800/2351 | Loss: 0.6288
  Epoch 1 | Batch 1900/2351 | Loss: 0.7479
  Epoch 1 | Batch 2000/2351 | Loss: 0.6637
  Epoch 1 | Batch 2100/2351 | Loss: 0.6517
  Epoch 1 | Batch 2200/2351 | Loss: 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ GPT-2 model saved!


In [12]:
!pip install scikit-learn -q
print("✅ sklearn installed!")

✅ sklearn installed!


In [13]:
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

print("🧪 Testing fine-tuned GPT-2 model...")
model2.eval()
torch.cuda.empty_cache()

def generate_response(question, max_new_tokens=100):
    text   = f"Customer: {question} Support:"
    inputs = tokenizer2(text, return_tensors="pt",
                       truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model2.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer2.eos_token_id,
            eos_token_id=tokenizer2.eos_token_id,
        )
    full_response = tokenizer2.decode(outputs[0], skip_special_tokens=True)
    response      = full_response.split("Support:")[-1].strip()
    return response if response else "I can help you with that."

# Test with 10 real questions from dataset
test_samples = df_clean.sample(10, random_state=42)
vectorizer   = TfidfVectorizer()

print("\n" + "="*60)
print("   SIMILARITY TEST RESULTS")
print("="*60)

similarities = []
for i, (_, row) in enumerate(test_samples.iterrows()):
    question  = row['instruction']
    expected  = row['response']

    try:
        generated = generate_response(question)
    except Exception as e:
        generated = "I can help you with that."

    tfidf_matrix = vectorizer.fit_transform([expected, generated])
    similarity   = cosine_similarity(
                    tfidf_matrix[0:1],
                    tfidf_matrix[1:2])[0][0]
    similarities.append(similarity)

    print(f"\nTest {i+1}:")
    print(f"  Question  : {question[:80]}")
    print(f"  Expected  : {expected[:80]}")
    print(f"  Generated : {generated[:80]}")
    print(f"  Similarity: {similarity:.2%}")

avg_similarity = np.mean(similarities)
print("\n" + "="*60)
print(f"  Average Similarity : {avg_similarity:.2%}")
print(f"  Min Similarity     : {min(similarities):.2%}")
print(f"  Max Similarity     : {max(similarities):.2%}")
print(f"  Target             : 60-70%")
print(f"  Status             : {'✅ PASSED' if avg_similarity >= 0.6 else '⚠️ Below target'}")
print("="*60)

🧪 Testing fine-tuned GPT-2 model...

   SIMILARITY TEST RESULTS

Test 1:
  Question  : I do not have a {{Account Type}} account, how do I register?
  Expected  : That's great to hear that you're interested in registering for a {{Account Type}
  Generated : Thank you for expressing your interest in registering for a {{Account Type}} acc
  Similarity: 50.96%

Test 2:
  Question  : I need help to cancel the corporate newsletter subscription
  Expected  : I'm cognizant of the fact that you would like assistance in canceling your corpo
  Generated : I've realized that you would like assistance with canceling your corporate newsl
  Similarity: 68.38%

Test 3:
  Question  : can utell me about the delivery periods
  Expected  : We completely understand your curiosity about delivery periods. Delivery periods
  Generated : We understand your curiosity about the delivery periods for your order. To provi
  Similarity: 62.67%

Test 4:
  Question  : I have problems with the removal of a standard acc

In [14]:
import shutil
shutil.make_archive("/workspace/gpt2-finetuned-backup", 'zip', "/workspace/gpt2-finetuned")
print("✅ Model zipped!")

✅ Model zipped!
